# Ordinary Kriging — k-fold CV Training Notebook

**What this notebook does (in order):**
1. Loads all wet-station records (1961–2023, 1966 train stations; 492 holdout excluded by `ReKISSource.load`).
2. Downloads the shared 5-fold station split from `s3://thesis-data-ismaktam/lgbm/fold_assignment.parquet` (canonical source — created by LGBM, also consumed by BayesNF).
3. Fits **one global climatological variogram** on all 1966 train stations (Haylock 2008 §3.3 literal). Cached to S3.
4. For each fold k: fits leakage-free TPS monthly normals on train-fold stations, predicts at test-fold station coords for all 9 (transform × variogram-model) combos, uploads `fold{k}_predictions.parquet` to S3.
5. Aggregates per-fold metrics into `cv_results.json` and uploads to S3.

**S3 layout:**
```
s3://thesis-data-ismaktam/kriging/
  global_variogram.pkl
  monthly_norms/
    fold{k}_norms_at_test_points.pkl
  kfold/
    fold{k}_predictions.parquet
    cv_results.json
```
**Local layout:** `outputs/kriging/` mirrors S3 (cache; downloaded on demand).

**Resume-safe:** every artefact tries local cache → S3 download → fresh compute (skip with `FORCE_RECOMPUTE = True`).

## 0. Imports

In [1]:
import os, sys, gc, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import boto3
from joblib import Parallel, delayed
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print(f'Working directory: {Path.cwd()}')
print(f'NumPy:    {np.__version__}')
print(f'Pandas:   {pd.__version__}')

Working directory: /root/precip_interpolation_thesis/notebooks/03_kriging
NumPy:    1.26.4
Pandas:   2.1.4


## 1. Constants + paths

In [2]:
# ── Reproducibility ───────────────────────────────────────────────────────
RANDOM_SEED = 42

# ── Cross-validation ─────────────────────────────────────────────────────
K_FOLDS    = 5
TRANSFORMS = ['none', 'log', 'normal_score']
VGM_MODELS = ['spherical', 'exponential', 'gaussian']

# ── MC back-transform ────────────────────────────────────────────────────
K_MC = 100

# ── Compute ──────────────────────────────────────────────────────────────
N_JOBS = int(os.environ.get('N_JOBS', '-1'))

# ── Prototyping limit (set None for full run) ────────────────────────────
N_TEST_DAYS = int(os.environ['N_TEST_DAYS']) if os.environ.get('N_TEST_DAYS') else None

# ── Local cache paths (mirror S3) ────────────────────────────────────────
OUT_DIR     = Path('outputs/kriging')
NORMS_DIR   = OUT_DIR / 'monthly_norms'
KFOLD_DIR   = OUT_DIR / 'kfold'
for d in (OUT_DIR, NORMS_DIR, KFOLD_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── S3 ───────────────────────────────────────────────────────────────────
S3_BUCKET = 'thesis-data-ismaktam'
S3_ROOT   = 'kriging'
FOLD_S3   = 'lgbm/fold_assignment.parquet'   # canonical source — LGBM

# Set True to ignore all local/S3 caches and redo everything from scratch
FORCE_RECOMPUTE = False

print(f'N_JOBS={N_JOBS}  N_TEST_DAYS={N_TEST_DAYS}  FORCE_RECOMPUTE={FORCE_RECOMPUTE}')
print(f'Local cache: {OUT_DIR}')
print(f'S3:          s3://{S3_BUCKET}/{S3_ROOT}/')

N_JOBS=-1  N_TEST_DAYS=None  FORCE_RECOMPUTE=False
Local cache: outputs/kriging
S3:          s3://thesis-data-ismaktam/kriging/


## 2. S3 helpers

In [3]:
s3 = boto3.client('s3')

def s3_exists(s3_key: str) -> bool:
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=s3_key)
        return True
    except Exception:
        return False

def s3_upload(local_path: Path, s3_key: str) -> None:
    try:
        s3.upload_file(str(local_path), S3_BUCKET, s3_key)
        print(f'  ↑ s3://{S3_BUCKET}/{s3_key}')
    except Exception as e:
        print(f'  S3 upload failed: {e.__class__.__name__}: {e}')

def s3_download(s3_key: str, local_path: Path) -> bool:
    try:
        local_path.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(S3_BUCKET, s3_key, str(local_path))
        print(f'  ↓ s3://{S3_BUCKET}/{s3_key}')
        return True
    except Exception:
        return False

def fetch(s3_key: str, local_path: Path) -> bool:
    """Ensure local_path exists. Returns True if available (local or via S3).
    Pattern: local cache → S3 → False."""
    if local_path.exists() and not FORCE_RECOMPUTE:
        return True
    if FORCE_RECOMPUTE:
        return False
    return s3_download(s3_key, local_path)

## 3. Load data + fit transform pipeline

Reuses `load_and_fit_pipeline` from `thesis.scripts._common` — same builder used by old scripts. Returns:
- `all_proc` — flat DataFrame with `Projection → Indicator → Detrend` applied
- `fwd`, `inv` — picklable transform callables
- `proc_by_date` — `{date: per-day DataFrame}`
- `det` — fitted `DetrendTransform` (needed for monthly normals)

In [4]:
_NB = Path.cwd()
while _NB != _NB.parent and not (_NB / 'pyproject.toml').exists():
    _NB = _NB.parent
if str(_NB / 'src') not in sys.path:
    sys.path.insert(0, str(_NB / 'src'))
os.chdir(_NB)
print(f'Project root: {_NB}')

from thesis.config import Config
from thesis.data.registry import DataRegistry
from thesis.scripts._common import load_and_fit_pipeline
from thesis.transforms import DetrendTransform, ProjectionTransform, IndicatorTransform
from thesis.transforms.pipeline import TransformPipeline

cfg      = Config()
registry = DataRegistry.from_config(cfg)

all_raw, all_proc, fwd, inv, proc_by_date, get_monthly_total = load_and_fit_pipeline(
    cfg, registry, cfg.date_start, cfg.date_end,
)

# Recover a fitted DetrendTransform handle (needed for build_monthly_norms_at_points)
det = DetrendTransform()
TransformPipeline([
    ProjectionTransform(target_crs=cfg.study_area.target_crs),
    IndicatorTransform(threshold_mm=cfg.wet_day_threshold_mm),
    det,
]).fit(all_raw)

print(f'all_proc rows : {len(all_proc):,}')
print(f'unique dates  : {len(proc_by_date):,}')
print(f'unique stations: {all_proc["station_id"].nunique():,}  (holdout 492 already excluded)')

Project root: /root/precip_interpolation_thesis
[01:19:38] Loading raw data: 1961-01-01 … 2023-12-31
[01:19:50]   45,237,660 rows, 1966 stations
[01:19:50] Fitting base pipeline: Projection → Indicator → Detrend
[01:20:08]   Fitting NormalScoreTransform…
[01:20:10]   NormalScoreTransform CDF: 17,409,376 wet quotas
all_proc rows : 45,237,660
unique dates  : 23,010
unique stations: 1,966  (holdout 492 already excluded)


## 4. Load shared fold assignment (from S3)

Reads `s3://thesis-data-ismaktam/lgbm/fold_assignment.parquet` — the canonical fold split created by LGBM and reused by BayesNF. Same five folds for all three models guarantees fair comparison.

In [5]:
FOLD_LOCAL = OUT_DIR / 'fold_assignment.parquet'
if not fetch(FOLD_S3, FOLD_LOCAL):
    raise FileNotFoundError(
        f'fold_assignment.parquet not in S3 at s3://{S3_BUCKET}/{FOLD_S3} — run lgbm_train.ipynb first to create it.'
    )

fold_assignment = pd.read_parquet(FOLD_LOCAL)
fold_map = dict(zip(fold_assignment['station_id'], fold_assignment['fold']))

missing = set(all_proc['station_id'].unique()) - set(fold_map)
print(f'Stations in all_proc       : {all_proc["station_id"].nunique()}')
print(f'Stations in fold_assignment: {len(fold_assignment)}')
print(f'Missing in fold_map        : {len(missing)}  (should be 0)')
print()
print('Fold sizes:')
print(fold_assignment['fold'].value_counts().sort_index().to_string())

all_proc['fold'] = all_proc['station_id'].map(fold_map).astype('Int64')

Stations in all_proc       : 1966
Stations in fold_assignment: 1966
Missing in fold_map        : 0  (should be 0)

Fold sizes:
fold
0    394
1    393
2    393
3    393
4    393


## 5. Fit ONE global climatological variogram (Haylock literal)

Following Haylock 2008 §3.3 [31]: *single variogram for all days*. Pools empirical semi-variances across ~14k days × ~1500 wet stations per day. Fitted on **all 1966 train stations** (per-fold sensitivity verified separately in `kriging_eval.ipynb`).

Cached at `s3://.../kriging/global_variogram.pkl`.

In [8]:
VGM_LOCAL = OUT_DIR / 'global_variogram.pkl'
VGM_S3    = f'{S3_ROOT}/global_variogram.pkl'

if fetch(VGM_S3, VGM_LOCAL):
    print(f'Loading cached variograms: {VGM_LOCAL}')
    with open(VGM_LOCAL, 'rb') as f:
        payload = pickle.load(f)
else:
    from thesis.models.kriging.variogram_fitter import GlobalVariogramFitter

    pool_dates = sorted(proc_by_date.keys())
    pool_procs = [proc_by_date[d] for d in pool_dates if (proc_by_date[d]['rain_indicator'] == 1).sum() >= 5]
    print(f'Pool ready: {len(pool_procs)} days')

    fitter = GlobalVariogramFitter(
        transforms       = TRANSFORMS,
        variogram_models = VGM_MODELS,
        n_lags           = cfg.kriging.variogram_nlags,
        max_lag_km       = cfg.kriging.search_radius_km,
        min_pairs        = 30,
        checkpoint_path  = str(OUT_DIR / 'global_variogram_checkpoint.pkl'),
        n_jobs           = N_JOBS,
    )
    fitter.fit(pool_procs, fwd_fn=fwd)

    print('Fitting indicator variogram (spherical)...')
    all_pool_procs = [proc_by_date[d] for d in pool_dates]
    fitter.fit_indicator(all_pool_procs)

    fitter.save(str(VGM_LOCAL))
    s3_upload(VGM_LOCAL, VGM_S3)
    with open(VGM_LOCAL, 'rb') as f:
        payload = pickle.load(f)

# Unwrap full payload → inner variogram dict with (transform, model) tuple keys
global_vgm = payload['variograms'] if isinstance(payload, dict) and 'variograms' in payload else payload

print('\n=== FITTED VARIOGRAMS ===')
for (t, vm), info in global_vgm.items():
    if info is None:
        print(f'  ({t!r:<14}, {vm!r:<12}) -> FAILED')
    else:
        p = info['params_dict']
        print(f'  ({t!r:<14}, {vm!r:<12}) nugget={p["nugget"]:.3f}  psill={p["psill"]:.3f}  range={p["range"]/1000:>6.1f} km')

Loading cached variograms: outputs/kriging/global_variogram.pkl

=== FITTED VARIOGRAMS ===
  ('none'        , 'spherical' ) nugget=0.002  psill=0.006  range= 216.3 km
  ('none'        , 'exponential') nugget=0.000  psill=0.008  range=  78.7 km
  ('none'        , 'gaussian'  ) nugget=0.003  psill=0.005  range= 104.8 km
  ('log'         , 'spherical' ) nugget=0.251  psill=0.467  range= 327.9 km
  ('log'         , 'exponential') nugget=0.172  psill=0.614  range= 149.4 km
  ('log'         , 'gaussian'  ) nugget=0.311  psill=0.398  range= 152.5 km
  ('normal_score', 'spherical' ) nugget=0.268  psill=0.489  range= 345.0 km
  ('normal_score', 'exponential') nugget=0.191  psill=0.645  range= 163.3 km
  ('normal_score', 'gaussian'  ) nugget=0.330  psill=0.414  range= 159.2 km
  ('indicator'   , 'spherical' ) nugget=0.059  psill=0.094  range= 416.0 km


## 6. Per-fold monthly TPS normals

Fitted on train-fold stations only (~1573 stations per fold), evaluated at fold-k test station coords. Uploaded to `s3://.../kriging/monthly_norms/fold{k}_norms_at_test_points.pkl`. Parallelised over folds (5 workers — folds are independent).

In [9]:
from thesis.models.kriging.monthly_norms import build_monthly_norms_at_points

def _fit_fold_norms_at_points(k, all_proc_df, det_obj, out_dir, s3_root, force):
    out_local = out_dir / f'fold{k}_norms_at_test_points.pkl'
    s3_key    = f'{s3_root}/monthly_norms/fold{k}_norms_at_test_points.pkl'
    # local cache
    if out_local.exists() and not force:
        return k, out_local, 'cache'
    # try S3
    if not force:
        s3c = boto3.client('s3')
        try:
            out_local.parent.mkdir(parents=True, exist_ok=True)
            s3c.download_file(S3_BUCKET, s3_key, str(out_local))
            return k, out_local, 's3'
        except Exception:
            pass

    train_mask = all_proc_df['fold'] != k
    test_mask  = all_proc_df['fold'] == k
    train_proc = all_proc_df[train_mask]
    test_meta = (
        all_proc_df[test_mask][['station_id', 'x_proj', 'y_proj'] + (['elevation_m'] if 'elevation_m' in all_proc_df.columns else [])]
        .drop_duplicates('station_id')
        .reset_index(drop=True)
    )
    elev = test_meta['elevation_m'].values if 'elevation_m' in test_meta.columns else None
    norms_2d, norms_3d = build_monthly_norms_at_points(
        det_obj, train_proc,
        test_meta['x_proj'].values, test_meta['y_proj'].values, elev,
    )
    payload = {
        'fold': k,
        'station_ids': test_meta['station_id'].values,
        'norms_2d':    norms_2d,
        'norms_3d':    norms_3d,
    }
    with open(out_local, 'wb') as f:
        pickle.dump(payload, f)
    return k, out_local, 'fresh'

n_workers = min(K_FOLDS, abs(N_JOBS) if N_JOBS != -1 else os.cpu_count())
fold_norm_results = Parallel(n_jobs=n_workers, backend='loky')(
    delayed(_fit_fold_norms_at_points)(k, all_proc, det, NORMS_DIR, S3_ROOT, FORCE_RECOMPUTE)
    for k in range(K_FOLDS)
)
for k, p, src in fold_norm_results:
    print(f'  fold {k}: {p}  [{src}]')
    if src == 'fresh':
        s3_upload(p, f'{S3_ROOT}/monthly_norms/fold{k}_norms_at_test_points.pkl')

  fold 0: outputs/kriging/monthly_norms/fold0_norms_at_test_points.pkl  [cache]
  fold 1: outputs/kriging/monthly_norms/fold1_norms_at_test_points.pkl  [cache]
  fold 2: outputs/kriging/monthly_norms/fold2_norms_at_test_points.pkl  [cache]
  fold 3: outputs/kriging/monthly_norms/fold3_norms_at_test_points.pkl  [cache]
  fold 4: outputs/kriging/monthly_norms/fold4_norms_at_test_points.pkl  [cache]


## 7. K-fold CV: parallel per-day prediction loop

For each fold k: train kriging on fold≠k stations, predict at fold==k coords for all 9 (transform × variogram) combos. Saves `fold{k}_predictions.parquet` after each fold (resume-safe) and uploads to S3.

In [10]:
from thesis.models.kriging.kfold_worker import predict_one_day_all_combos

USE_3D_NORMS  = ('elevation_m' in all_proc.columns) and all_proc['elevation_m'].notna().any()
MAX_WET        = cfg.kriging.max_wet
N_STATIONS_MIN = cfg.kriging.n_stations_min

# Rebuild proc_by_date AFTER fold column was added to all_proc (cell 4)
proc_by_date = {str(d): grp for d, grp in all_proc.groupby('date')}

all_dates_sorted = sorted(proc_by_date.keys())
if N_TEST_DAYS is not None:
    rng = np.random.default_rng(RANDOM_SEED)
    sampled_dates = sorted(rng.choice(all_dates_sorted, size=min(N_TEST_DAYS, len(all_dates_sorted)), replace=False).tolist())
else:
    sampled_dates = all_dates_sorted
print(f'Days to predict: {len(sampled_dates):,}')

for k in range(K_FOLDS):
    out_local = KFOLD_DIR / f'fold{k}_predictions.parquet'
    out_s3    = f'{S3_ROOT}/kfold/fold{k}_predictions.parquet'

    if fetch(out_s3, out_local):
        print(f'\n[fold {k}] Cached: {out_local} — skipping')
        continue

    print(f'\n[fold {k}] Building train/test splits...')
    norms_local = NORMS_DIR / f'fold{k}_norms_at_test_points.pkl'
    if not fetch(f'{S3_ROOT}/monthly_norms/fold{k}_norms_at_test_points.pkl', norms_local):
        raise RuntimeError(f'Fold {k} monthly norms missing — run cell 6 first')
    with open(norms_local, 'rb') as f:
        fold_norms = pickle.load(f)
    test_sids = fold_norms['station_ids']
    norms_2d  = fold_norms['norms_2d']
    norms_3d  = fold_norms['norms_3d']
    test_sid_to_idx = {s: i for i, s in enumerate(test_sids)}

    proc_train_test = []
    for date in sampled_dates:
        proc_d = proc_by_date[date]
        is_test = proc_d['fold'] == k
        proc_test_d  = proc_d[is_test]
        proc_train_d = proc_d[~is_test]
        if len(proc_test_d) > 0:
            test_idx = np.array([test_sid_to_idx[s] for s in proc_test_d['station_id'].values])
            n2d_today = norms_2d[:, test_idx]
            n3d_today = norms_3d[:, test_idx] if norms_3d is not None else None
        else:
            n2d_today = None
            n3d_today = None
        proc_train_test.append((date, proc_train_d, proc_test_d, n2d_today, n3d_today))

    print(f'[fold {k}] Predicting {len(proc_train_test)} days × {len(TRANSFORMS) * len(VGM_MODELS)} combos...')
    fold_results = Parallel(n_jobs=N_JOBS, backend='loky', verbose=5)(
        delayed(predict_one_day_all_combos)(
            date, proc_train_d, proc_test_d,
            global_vgm, TRANSFORMS, VGM_MODELS,
            fwd, inv, n2d_today, n3d_today,
            N_STATIONS_MIN, MAX_WET, K_MC, RANDOM_SEED,
            USE_3D_NORMS,
        )
        for date, proc_train_d, proc_test_d, n2d_today, n3d_today in proc_train_test
    )

    fold_df = pd.concat([df for df in fold_results if not df.empty], ignore_index=True)
    fold_df['fold'] = k
    fold_df.to_parquet(out_local, index=False)
    s3_upload(out_local, out_s3)
    print(f'[fold {k}] Saved {len(fold_df):,} records → {out_local}')
    del fold_results, fold_df, proc_train_test
    gc.collect()

Days to predict: 23,010

[fold 0] Cached: outputs/kriging/kfold/fold0_predictions.parquet — skipping

[fold 1] Building train/test splits...
[fold 1] Predicting 23010 days × 9 combos...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 185 concurrent workers.
[Parallel(n_jobs=-1)]: Done  80 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 278 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done 512 tasks      | elapsed:  3.4min
[Parallel(n_jobs=-1)]: Done 782 tasks      | elapsed:  4.9min
[Parallel(n_jobs=-1)]: Done 1088 tasks      | elapsed:  6.5min
[Parallel(n_jobs=-1)]: Done 1430 tasks      | elapsed:  8.4min
[Parallel(n_jobs=-1)]: Done 1808 tasks      | elapsed: 10.6min
[Parallel(n_jobs=-1)]: Done 2222 tasks      | elapsed: 12.9min
[Parallel(n_jobs=-1)]: Done 2672 tasks      | elapsed: 15.4min
[Parallel(n_jobs=-1)]: Done 3158 tasks      | elapsed: 18.1min
[Parallel(n_jobs=-1)]: Done 3680 tasks      | elapsed: 21.0min
[Parallel(n_jobs=-1)]: Done 4238 tasks      | elapsed: 24.0min
[Parallel(n_jobs=-1)]: Done 4832 tasks      | elapsed: 27.2min
[Parallel(n_jobs=-1)]: Done 5462 tasks      | elapsed: 30.7min
[Parallel(n_jobs=-1)]: Done 6128 tasks      

  ↑ s3://thesis-data-ismaktam/kriging/kfold/fold1_predictions.parquet
[fold 1] Saved 81,386,370 records → outputs/kriging/kfold/fold1_predictions.parquet

[fold 2] Building train/test splits...
[fold 2] Predicting 23010 days × 9 combos...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 185 concurrent workers.
[Parallel(n_jobs=-1)]: Done  80 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done 278 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done 512 tasks      | elapsed:  3.4min
[Parallel(n_jobs=-1)]: Done 782 tasks      | elapsed:  4.9min
[Parallel(n_jobs=-1)]: Done 1088 tasks      | elapsed:  6.5min
[Parallel(n_jobs=-1)]: Done 1430 tasks      | elapsed:  8.4min
[Parallel(n_jobs=-1)]: Done 1808 tasks      | elapsed: 10.6min
[Parallel(n_jobs=-1)]: Done 2222 tasks      | elapsed: 12.9min
[Parallel(n_jobs=-1)]: Done 2672 tasks      | elapsed: 15.4min
[Parallel(n_jobs=-1)]: Done 3158 tasks      | elapsed: 18.1min
[Parallel(n_jobs=-1)]: Done 3680 tasks      | elapsed: 21.0min
[Parallel(n_jobs=-1)]: Done 4238 tasks      | elapsed: 24.0min
[Parallel(n_jobs=-1)]: Done 4832 tasks      | elapsed: 27.2min
[Parallel(n_jobs=-1)]: Done 5462 tasks      | elapsed: 30.8min
[Parallel(n_jobs=-1)]: Done 6128 tasks      

  ↑ s3://thesis-data-ismaktam/kriging/kfold/fold2_predictions.parquet
[fold 2] Saved 81,386,370 records → outputs/kriging/kfold/fold2_predictions.parquet

[fold 3] Building train/test splits...
[fold 3] Predicting 23010 days × 9 combos...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 185 concurrent workers.
[Parallel(n_jobs=-1)]: Done  80 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done 278 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done 512 tasks      | elapsed:  3.4min
[Parallel(n_jobs=-1)]: Done 782 tasks      | elapsed:  4.9min
[Parallel(n_jobs=-1)]: Done 1088 tasks      | elapsed:  6.5min
[Parallel(n_jobs=-1)]: Done 1430 tasks      | elapsed:  8.4min
[Parallel(n_jobs=-1)]: Done 1808 tasks      | elapsed: 10.6min
[Parallel(n_jobs=-1)]: Done 2222 tasks      | elapsed: 12.9min
[Parallel(n_jobs=-1)]: Done 2672 tasks      | elapsed: 15.4min
[Parallel(n_jobs=-1)]: Done 3158 tasks      | elapsed: 18.1min
[Parallel(n_jobs=-1)]: Done 3680 tasks      | elapsed: 21.0min
[Parallel(n_jobs=-1)]: Done 4238 tasks      | elapsed: 24.1min
[Parallel(n_jobs=-1)]: Done 4832 tasks      | elapsed: 27.3min
[Parallel(n_jobs=-1)]: Done 5462 tasks      | elapsed: 30.9min
[Parallel(n_jobs=-1)]: Done 6128 tasks      

  ↑ s3://thesis-data-ismaktam/kriging/kfold/fold3_predictions.parquet
[fold 3] Saved 81,386,370 records → outputs/kriging/kfold/fold3_predictions.parquet

[fold 4] Building train/test splits...
[fold 4] Predicting 23010 days × 9 combos...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 185 concurrent workers.
[Parallel(n_jobs=-1)]: Done  80 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done 278 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done 512 tasks      | elapsed:  3.4min
[Parallel(n_jobs=-1)]: Done 782 tasks      | elapsed:  4.9min
[Parallel(n_jobs=-1)]: Done 1088 tasks      | elapsed:  6.5min
[Parallel(n_jobs=-1)]: Done 1430 tasks      | elapsed:  8.4min
[Parallel(n_jobs=-1)]: Done 1808 tasks      | elapsed: 10.7min
[Parallel(n_jobs=-1)]: Done 2222 tasks      | elapsed: 13.0min
[Parallel(n_jobs=-1)]: Done 2672 tasks      | elapsed: 15.4min
[Parallel(n_jobs=-1)]: Done 3158 tasks      | elapsed: 18.2min
[Parallel(n_jobs=-1)]: Done 3680 tasks      | elapsed: 21.1min
[Parallel(n_jobs=-1)]: Done 4238 tasks      | elapsed: 24.0min
[Parallel(n_jobs=-1)]: Done 4832 tasks      | elapsed: 27.4min
[Parallel(n_jobs=-1)]: Done 5462 tasks      | elapsed: 30.9min
[Parallel(n_jobs=-1)]: Done 6128 tasks      

  ↑ s3://thesis-data-ismaktam/kriging/kfold/fold4_predictions.parquet
[fold 4] Saved 81,386,370 records → outputs/kriging/kfold/fold4_predictions.parquet


## 8. Aggregate per-fold metrics → cv_results.json (uploaded to S3)

In [11]:
import properscoring as ps

rows = []
for k in range(K_FOLDS):
    fold_local = KFOLD_DIR / f'fold{k}_predictions.parquet'
    if not fetch(f'{S3_ROOT}/kfold/fold{k}_predictions.parquet', fold_local):
        print(f'[fold {k}] missing — skipping aggregation')
        continue
    df = pd.read_parquet(fold_local)
    n_test_stations = df['station_id'].nunique()
    for (t, vm), grp in df.groupby(['transform', 'variogram_model']):
        obs  = grp['observed_mm'].values
        pred = grp['predicted_mm'].values
        sd   = np.sqrt(np.maximum(grp['predicted_var_mm2'].values, 1e-8))
        rows.append({
            'fold': int(k), 'transform': t, 'variogram_model': vm,
            'crps': float(np.mean(ps.crps_gaussian(obs, mu=pred, sig=sd))),
            'mae':  float(np.mean(np.abs(obs - pred))),
            'rmse': float(np.sqrt(np.mean((obs - pred) ** 2))),
            'n_test': int(len(grp)), 'n_test_stations': int(n_test_stations),
        })

cv_local = KFOLD_DIR / 'cv_results.json'
with open(cv_local, 'w') as f:
    json.dump(rows, f, indent=2)
s3_upload(cv_local, f'{S3_ROOT}/kfold/cv_results.json')

summary = pd.DataFrame(rows)
print('\n=== PER-FOLD METRICS ===')
print(summary.to_string(index=False))
print('\n=== MEAN ACROSS FOLDS ===')
print(summary.groupby(['transform', 'variogram_model'])[['crps', 'mae', 'rmse']].mean().round(4).to_string())

  ↑ s3://thesis-data-ismaktam/kriging/kfold/cv_results.json

=== PER-FOLD METRICS ===
 fold    transform variogram_model     crps      mae     rmse  n_test  n_test_stations
    0          log     exponential 0.450199 0.539911 1.483882 9065940              394
    0          log        gaussian 0.556024 0.697388 1.849880 9065940              394
    0          log       spherical 0.493294 0.590263 1.585460 9065940              394
    0         none     exponential 0.416698 0.523323 1.401063 9065940              394
    0         none        gaussian 0.545005 0.709457 1.746346 9065940              394
    0         none       spherical 0.474669 0.598119 1.499495 9065940              394
    0 normal_score     exponential 0.442050 0.535988 1.474041 9065940              394
    0 normal_score        gaussian 0.543575 0.693684 1.839840 9065940              394
    0 normal_score       spherical 0.478583 0.581361 1.564198 9065940              394
    1          log     exponential 0.443798 

## 9. Done — S3 manifest

In [12]:
print('=== S3 ARTEFACTS ===')
for k in range(K_FOLDS):
    print(f'  s3://{S3_BUCKET}/{S3_ROOT}/monthly_norms/fold{k}_norms_at_test_points.pkl  '
          f'{"✓" if s3_exists(f"{S3_ROOT}/monthly_norms/fold{k}_norms_at_test_points.pkl") else "✗"}')
    print(f'  s3://{S3_BUCKET}/{S3_ROOT}/kfold/fold{k}_predictions.parquet  '
          f'{"✓" if s3_exists(f"{S3_ROOT}/kfold/fold{k}_predictions.parquet") else "✗"}')
for key in ['global_variogram.pkl', 'kfold/cv_results.json']:
    print(f'  s3://{S3_BUCKET}/{S3_ROOT}/{key}  {"✓" if s3_exists(f"{S3_ROOT}/{key}") else "✗"}')

=== S3 ARTEFACTS ===
  s3://thesis-data-ismaktam/kriging/monthly_norms/fold0_norms_at_test_points.pkl  ✓
  s3://thesis-data-ismaktam/kriging/kfold/fold0_predictions.parquet  ✓
  s3://thesis-data-ismaktam/kriging/monthly_norms/fold1_norms_at_test_points.pkl  ✓
  s3://thesis-data-ismaktam/kriging/kfold/fold1_predictions.parquet  ✓
  s3://thesis-data-ismaktam/kriging/monthly_norms/fold2_norms_at_test_points.pkl  ✓
  s3://thesis-data-ismaktam/kriging/kfold/fold2_predictions.parquet  ✓
  s3://thesis-data-ismaktam/kriging/monthly_norms/fold3_norms_at_test_points.pkl  ✓
  s3://thesis-data-ismaktam/kriging/kfold/fold3_predictions.parquet  ✓
  s3://thesis-data-ismaktam/kriging/monthly_norms/fold4_norms_at_test_points.pkl  ✓
  s3://thesis-data-ismaktam/kriging/kfold/fold4_predictions.parquet  ✓
  s3://thesis-data-ismaktam/kriging/global_variogram.pkl  ✓
  s3://thesis-data-ismaktam/kriging/kfold/cv_results.json  ✓
